In [1]:
#Import pandas, matplotlib.pyplot, and seaborn
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [2]:
#Import the clean dataset 
df=pd.read_csv(r"../data/brfss_2021_clean.csv")

In [3]:
#Review the first few rows
df.head()

,_STATE,PERSDOC3,_RFHYPE6,MARITAL,EMPLOY1,_METSTAT,_URBSTAT,_RFHLTH,_PHYS14D,_HLTHPLN,...,_CURECI1_missing,_SMOKER3_missing,DRNKANY5_missing,GRENDA1__missing,FRNCHDA__missing,_FRUTSU1_missing,_VEGESU1_missing,_BMI5_missing,INCOME_GROUP,HIGHBP
0,1.0,1.0,0.0,1.0,7.0,1.0,1.0,2.0,3.0,1.0,...,0,0,0,0,0,0,0,0,Low,0
1,1.0,2.0,1.0,1.0,8.0,1.0,1.0,1.0,1.0,1.0,...,0,0,0,0,0,0,0,1,Missing,1
2,1.0,2.0,1.0,3.0,7.0,1.0,1.0,1.0,1.0,1.0,...,0,0,0,0,0,0,0,0,Low,1
3,1.0,1.0,1.0,1.0,7.0,1.0,1.0,1.0,1.0,1.0,...,0,0,0,0,0,0,0,0,Medium,1
4,1.0,1.0,0.0,1.0,8.0,2.0,1.0,2.0,3.0,1.0,...,0,0,0,0,0,0,0,0,Low,0


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 436781 entries, 0 to 436780
Data columns (total 39 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   _STATE            436781 non-null  float64
 1   PERSDOC3          436781 non-null  float64
 2   _RFHYPE6          436781 non-null  float64
 3   MARITAL           436781 non-null  float64
 4   EMPLOY1           436781 non-null  float64
 5   _METSTAT          436781 non-null  float64
 6   _URBSTAT          436781 non-null  float64
 7   _RFHLTH           436781 non-null  float64
 8   _PHYS14D          436781 non-null  float64
 9   _HLTHPLN          436781 non-null  float64
 10  _TOTINDA          436781 non-null  float64
 11  _RACE             436781 non-null  float64
 12  _SEX              436781 non-null  float64
 13  _AGEG5YR          383410 non-null  float64
 14  _AGE80            436781 non-null  float64
 15  _BMI5             436781 non-null  float64
 16  _BMI5CAT          39

In [5]:
df.shape

(436781, 39)

Based on my observations from the data wrangling and EDA, I will drop:

1. HIGHBP - this is your target variable, not a predictor. Will not be included in the features 
2. _RFHYPE6 - this is original hypertension variable. We have HIGHBP as the binary variable 
3. GRENDA1_ - showed moderate correlation with other diet variable and would be redundant 
4. DRNKANY5 - EDA showed some doubts about the truthfullness of the responses  
5. GRENDA1__missing
6. DRNKANY5_missing
7. _AGEG5YR - keeping the continous variable _AGE80 as it has no missing values 
8. _BMI5CAT - keeping the continous variable _BMI5 as it has no missing values 
9. _INCOMG1 - since I created INCOME_GROUP
10. _STATE - will create too many dummy variables and reduce interpretability|

In [9]:
# Target variable
y = df['HIGHBP']

# Features
X = df.drop(columns=[
    'HIGHBP',
    '_RFHYPE6',
    'GRENDA1_',
    'GRENDA1__missing',
    'DRNKANY5',
    'DRNKANY5_missing',
    '_AGEG5YR',
    '_BMI5CAT',
    '_INCOMG1', 
    '_STATE'
])

In [11]:
print(X.shape)
print(y.shape)

(436781, 29)
(436781,)


In [13]:
#Splitting all the features into categorical, numerical, missing indicator 

categorical_features = [
    'PERSDOC3',
    'MARITAL',
    'EMPLOY1',
    '_METSTAT',
    '_URBSTAT',
    '_RFHLTH',
    '_HLTHPLN',
    '_TOTINDA',
    '_RACE',
    '_SEX',
    '_EDUCAG',
    '_SMOKER3',
    '_CURECI1',
    'INCOME_GROUP'
]

numerical_features = [
    '_AGE80',
    '_BMI5',
    '_PHYS14D',
    '_FRUTSU1',
    '_VEGESU1',
    'FRNCHDA_'
]

#Missing indicator features are already binary 
missing_indicator_features = [
    '_PHYS14D_missing',
    '_RACE_missing',
    '_HLTHPLN_missing',
    '_CURECI1_missing',
    '_SMOKER3_missing',
    'FRNCHDA__missing',
    '_FRUTSU1_missing',
    '_VEGESU1_missing',
    '_BMI5_missing'
]

In [15]:
#Use one-hot encoding to create dummy variables for all categorical variables

X = pd.get_dummies(
    X,
    columns=categorical_features,
    drop_first=True
)

In [17]:
print("Shape after one-hot encoding:", X.shape)

Shape after one-hot encoding: (436781, 53)


In [19]:
#Split the data into train/test 

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [21]:
print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

Training set: (349424, 53)
Testing set: (87357, 53)


In [23]:
#Checking class balance 

print("Training Set Class Distribution")
print(y_train.value_counts(normalize=True))

print("\nTesting Set Class Distribution")
print(y_test.value_counts(normalize=True))

Training Set Class Distribution
HIGHBP
0    0.605906
1    0.394094
Name: proportion, dtype: float64

Testing Set Class Distribution
HIGHBP
0    0.605905
1    0.394095
Name: proportion, dtype: float64


In [27]:
#Standardize the numerical variables for the training data

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train[numerical_features] = scaler.fit_transform(
    X_train[numerical_features]
)

X_test[numerical_features] = scaler.transform(
    X_test[numerical_features]
)

In [29]:
#Checking if the standardization worked 

print(X_train[numerical_features].describe().round(2))

          _AGE80      _BMI5   _PHYS14D   _FRUTSU1   _VEGESU1   FRNCHDA_
count  349424.00  349424.00  349424.00  349424.00  349424.00  349424.00
mean       -0.00       0.00       0.00      -0.00      -0.00       0.00
std         1.00       1.00       1.00       1.00       1.00       1.00
min        -2.09      -2.70      -0.63      -1.19      -1.47      -0.76
25%        -0.78      -0.66      -0.63      -0.65      -0.50      -0.50
50%         0.13      -0.16      -0.63      -0.27      -0.18      -0.23
75%         0.81       0.43       0.81       0.65       0.23       0.34
max         1.44       6.82       2.25       8.01      10.37      10.61


In [32]:
#Saving the current dataframe as a csv file
df.to_csv('../data/brfss_2021_processed.csv', index=False)